In [112]:
%pip install opencv-python numpy matplotlib imageio scikit-image>=0.25 scikit-learn ipywidgets ipykernel imageio[ffmpeg]

In [113]:
from matplotlib import pyplot as plt
import skimage
import numpy as np
import imageio.v3 as iio
from ipywidgets import interact, IntSlider
import cv2 as cv

In [114]:
focal = 485.82423388827533     # focal length (f)
centerx = 134.875              # principal point x (cx)
centery = 239.875              # principal point y (cy)
ball_radius = 3                # ball radius in centimeters (R)
video_path = 'videos/red_and_blue.mov'  # path to video

COLOR = (220, 20, 20)

In [115]:
frames = [frame for frame in iio.imiter(video_path)] # load all frames of video

In [116]:
from skimage.feature import canny

def preprocess_image(image):
    """ Convert image to grayscale and run Canny edge detector.
        Arguments:
            image: RGB image
        Returns:
            Result of Canny edge detector
    """
    # Convert to grayscale
    grayscale_img = skimage.color.rgb2gray(image)

    # Canny Edge Detector
    return canny(grayscale_img, sigma=1.5, low_threshold=0.2, high_threshold=0.7)


In [117]:
from skimage.transform import hough_circle, hough_circle_peaks

def detect_circles(edges, total_num_peaks=1):
    """ Detect circles in image.
        Arguments:
            edges: edge image produced by preprocess_image()
            total_num_peaks: the number of circles to detect (default to 1)
        Returns:
            List of tuples (x, y, radius)
    """
    hough_radii = np.arange(20, 35, 2)
    hough_res = hough_circle(edges, hough_radii)

    accums, cx, cy, radii = hough_circle_peaks(hough_res, hough_radii, total_num_peaks=total_num_peaks)

    return list(zip(cx, cy, radii))


In [118]:
def calculate_ball_position(x,y,r):
    """ Calculate ball's (X,Y,Z) position in world coordinates
        Arguments:
            x,y: 2D position of ball in image
            r: radius of ball in image
        Returns:
            X,Y,Z position of ball in world coordinates
    """
    world_z = focal * (ball_radius/r)
    world_x = world_z * ((x-centerx)/focal)
    world_y = world_z * ((y-centery)/focal)

    return world_x, world_y, world_z


In [131]:
from skimage.draw import circle_perimeter

def draw_ball(x,y,r,Z, image):
    """ Draw circle on ball and write depth estimate in center
        Arguments:
            x,y,r: 2D position and radius of ball
            Z: estimated depth of ball
            image: the original frame to overlay the ball onto
        Returns:
            image: the original image with a new line overlayed on it
    """
    overlayed_image = np.copy(image)
    circy, circx = circle_perimeter(y, x, r, shape=overlayed_image.shape)
    overlayed_image[circy, circx] =  COLOR

    # Put the estimated depth at the center
    text = f"{Z:.2f}cm"
    font = cv.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    color = COLOR
    thickness = 3
    cv.putText(overlayed_image, text, (x, y), font, font_scale, color, thickness)

    return overlayed_image


In [120]:
def project(X,Y,Z):
    """ Pinhole projection.
        Arguments:
            X,Y,Z: 3D point
        Returns:
            (x,y) 2D location of projection in image
    """
    image_x = focal * (X / Z) + centerx
    image_y = focal * (Y / Z) + centery

    return image_x, image_y

In [121]:
def draw_line_2d(x1,y1,x2,y2, image):
    """ Draw a 2D line
        Arguments:
            x1,y1: 2D position of first line endpoint
            x2,y2: 2D position of second line endpoint
            image: the original frame to overlay the ball onto
        Returns:
            image: the original image with a new line overlayed on it
    """
    overlayed_image = np.copy(image)

    return cv.line(overlayed_image, (x1, y1), (x2, y2), COLOR)

In [122]:
def draw_line_3d(X1,Y1,Z1,X2,Y2,Z2, image):
    """ Draw a 3D line.  (This should call draw_line_2d.)
        Arguments:
            X1,Y1,Z1: 3D position of first line endpoint
            X2,Y2,Z2: 3D position of second line endpoint
        Returns:
            image: the original image with a new line overlayed on it
    """
    # Convert the 3d points to 2d
    x1, y1 = project(X1, Y1, Z1)
    x2, y2 = project(X2, Y2, Z2)

    # Overlay the points using draw_line_2d
    return draw_line_2d(int(x1), int(y1), int(x2), int(y2), image)


In [123]:
def draw_bounding_cube(X,Y,Z, image):
    """ Draw bounding cube around 3D point, with radius R
        Arguments:
            image: image on which to draw
            X,Y,Z: 3D center point of cube
            image: the original frame to overlay the ball onto
        Returns:
            image: the original image with a new line overlayed on it
    """
    # Get the 8 points around the center of the cube (in 3d)
    """
       H --- G
      /|    /|
     E --- F |
     | |   | |
     | D --|- C
     |/    |/
     A --- B
    """
    r = ball_radius
    ax, ay, az = X-r, Y+r, Z-r
    bx, by, bz = X+r, Y+r, Z-r
    cx, cy, cz = X+r, Y-r, Z-r
    dx, dy, dz = X-r, Y-r, Z-r
    ex, ey, ez = X-r, Y+r, Z+r
    fx, fy, fz = X+r, Y+r, Z+r
    gx, gy, gz = X+r, Y-r, Z+r
    hx, hy, hz = X-r, Y-r, Z+r

    # draw_line_3d from these points
    overlayed_image = np.copy(image)
    # a-b
    overlayed_image = draw_line_3d(ax, ay, az, bx, by, bz, overlayed_image)

    # a-e
    overlayed_image = draw_line_3d(ax, ay, az, ex, ey, ez, overlayed_image)

    # a-d
    overlayed_image = draw_line_3d(ax, ay, az, dx, dy, dz, overlayed_image)

    # f-e
    overlayed_image = draw_line_3d(fx, fy, fz, ex, ey, ez, overlayed_image)

    # f-g
    overlayed_image = draw_line_3d(fx, fy, fz, gx, gy, gz, overlayed_image)

    # f-b
    overlayed_image = draw_line_3d(fx, fy, fz, bx, by, bz, overlayed_image)

    # c-b
    overlayed_image = draw_line_3d(cx, cy, cz, bx, by, bz, overlayed_image)

    # c-d
    overlayed_image = draw_line_3d(cx, cy, cz, dx, dy, dz, overlayed_image)

    # c-g
    overlayed_image = draw_line_3d(cx, cy, cz, gx, gy, gz, overlayed_image)

    # h-e
    overlayed_image = draw_line_3d(hx, hy, hz, ex, ey, ez, overlayed_image)

    # h-g
    overlayed_image = draw_line_3d(hx, hy, hz, gx, gy, gz, overlayed_image)

    # h-d
    overlayed_image = draw_line_3d(hx, hy, hz, dx, dy, dz, overlayed_image)

    return overlayed_image



In [132]:
# --- Full Demo ---
# Show the result of the preprocessed image
def show_preprocessed_image(i):
  frame = np.copy(frames[i])

  edges = preprocess_image(frame)
  plt.imshow(edges)

interact(show_preprocessed_image, i=IntSlider(min=0, max=len(frames)-1, step=1, value=0))

# Show the result of detect_circles and calculate_ball_position
# (use draw_ball to draw the ball on the image.)
# You should draw all balls detected, even if there is more than one.
def show_preprocessed_image(i):
  frame = np.copy(frames[i])

  edges = preprocess_image(frame)
  for (cx, cy, r) in detect_circles(edges, 2):
    x, y, z = calculate_ball_position(cx, cy, r)
    frame = draw_ball(cx, cy, r, z, frame)
  plt.imshow(frame, cmap="gray")

interact(show_preprocessed_image, i=IntSlider(min=0, max=len(frames)-1, step=1, value=0))

# Show the reuslt of draw_bounding_cube
def show_preprocessed_image(i):
  frame = np.copy(frames[i])

  edges = preprocess_image(frame)
  for (cx, cy, r) in detect_circles(edges, 2):
    x, y, z = calculate_ball_position(cx, cy, r)
    frame = draw_bounding_cube(x, y, z, frame)
  plt.imshow(frame)

interact(show_preprocessed_image, i=IntSlider(min=0, max=len(frames)-1, step=1, value=0))

# Finally, concatenate all 3D ball positions detected across all frames into an
# array and show the result as a 3D scatter plot.


interactive(children=(IntSlider(value=0, description='i', max=338), Output()), _dom_classes=('widget-interact'…

interactive(children=(IntSlider(value=0, description='i', max=338), Output()), _dom_classes=('widget-interact'…

interactive(children=(IntSlider(value=0, description='i', max=338), Output()), _dom_classes=('widget-interact'…

<function __main__.show_preprocessed_image(i)>